# Q01 — Why do different bodies of water have different colours?

**Curriculum Level 1 (Observation)** · **Status:** `exploratory` (CPU-only notebook; configure local dataset paths in gitignored `config/local_paths.yaml` before executing).  
**Scope:** Computes GLORIA CIE 1931 2° chromaticity, Forel–Ule hue angle, AVW, and QWIP screening alongside NASA PACE OCI Level-2 AOP visible/false-colour RGB composites, swath PCA, trapezoidal AVW, CIE water colour, exploratory SAM classification against GLORIA AVW-quartile optical archetypes, and site-aggregated GLORIA-to-PACE SAM nearest-neighbour OOD screening.

**Scientific cautions:**
- Optical water classes and perceived water colour reflect mixtures of absorption and scattering (phytoplankton pigments, CDOM, mineral sediment, pure water). Do not equate an optical class or CIE hue angle with a taxonomic assemblage or constituent concentration.
- Interpolating GLORIA 1 nm spectra onto PACE OCI band centers (`np.interp`) is an exploratory alignment, not an official PACE OCI spectral response function (SRF) convolution.

In [ ]:
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

from hydrospectral.color import (
    chromaticity_to_hue_angle_deg,
    classify_owt_by_sam,
    rrs_to_xyz,
    xyz_to_chromaticity,
    xyz_to_display_srgb,
)
from hydrospectral.gloria import build_site_group_keys, extract_rrs_matrix, load_gloria_tables, summarize_qc_flags
from hydrospectral.manifests import load_active_sources
from hydrospectral.pace import (
    DEFAULT_PACE_EXCLUDE_FLAGS,
    build_rgb_composite,
    compute_swath_pca,
    decode_l2_flags,
    load_pace_l2_aop,
)
from hydrospectral.provenance import CohortCounts, build_manifest, sha256_file, write_manifest
from hydrospectral.radiometry import compute_avw, compute_qwip_score
from hydrospectral.splits import nearest_neighbor_sam_ood

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
LOCAL_PATHS_FILE = REPO_ROOT / 'config' / 'local_paths.yaml'
assert LOCAL_PATHS_FILE.is_file(), (
    'Copy config/local_paths.example.yaml to config/local_paths.yaml and set '
    'datasets.gloria_2022_zip, datasets.pace_oci_20240730_nrt, and workspace.scratch_dir.'
)
local_cfg = yaml.safe_load(LOCAL_PATHS_FILE.read_text(encoding='utf-8'))
sources = load_active_sources(REPO_ROOT / 'manifests' / 'sources')

GLORIA_PATH = Path(local_cfg['datasets']['gloria_2022_zip'])
PACE_PATH = Path(local_cfg['datasets']['pace_oci_20240730_nrt'])
OUT_DIR = Path(local_cfg['workspace']['scratch_dir']) / 'Q01_water_colour'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Loaded active source manifests:', sorted(sources.keys()))

## 1. GLORIA spectral quality, AVW, QWIP, and CIE 1931 chromaticity atlas

In [ ]:
gloria = load_gloria_tables(
    GLORIA_PATH,
    expected_sha256=sources['src.gloria-2022']['sha256'],
    include_qc_flags=True,
)
qc_summary = summarize_qc_flags(gloria.qc_flags)
wl_gloria, X_gloria = extract_rrs_matrix(gloria.rrs, min_nm=400, max_nm=700, step_nm=1)

complete_vis = np.isfinite(X_gloria).all(axis=1)
unflagged = ~qc_summary['any_flag'].to_numpy(dtype=bool)
n_total_gloria = len(gloria.meta)
n_incomplete_vis = int((~complete_vis).sum())
n_qc_flagged_complete = int((complete_vis & (~unflagged)).sum())
keep_vis = complete_vis & unflagged
X_vis = X_gloria[keep_vis]

# Preserve negative Rrs diagnostics rather than clipping.
n_neg_vis = int(np.any(X_vis < 0.0, axis=1).sum())
avw_nm = compute_avw(X_vis, wl_gloria, require_nonnegative=True)
qwip = compute_qwip_score(X_vis, wl_gloria, require_nonnegative_avw=True)
xyz_e = rrs_to_xyz(X_vis, wl_gloria, illuminant='E')
xy_e = xyz_to_chromaticity(xyz_e)
hue_deg = chromaticity_to_hue_angle_deg(xy_e)
srgb = xyz_to_display_srgb(xyz_e, illuminant='E', exposure='per_spectrum_max', gamut='desaturate')

gloria_color = pd.DataFrame({
    'GLORIA_ID': gloria.meta.loc[keep_vis, 'GLORIA_ID'].astype(str).to_numpy(),
    'avw_nm': avw_nm,
    'qwip_score': qwip['qwip_score'],
    'qwip_fail': qwip['qwip_fail'],
    'cie_x': xy_e[:, 0],
    'cie_y': xy_e[:, 1],
    'hue_angle_deg': hue_deg,
    'srgb_r': srgb[:, 0],
    'srgb_g': srgb[:, 1],
    'srgb_b': srgb[:, 2],
})
print(f'Retained unflagged complete 400-700 nm GLORIA spectra: {len(gloria_color)} (with >=1 negative Rrs: {n_neg_vis})')
gloria_color.describe()

## 2. PACE-01 Gulf of Mexico swath: flag screening, composites, PCA, AVW, and CIE colour

In [ ]:
# Optional: pass bbox=(-92.0, 26.0, -86.0, 30.5) to subset the northern Gulf swath on constrained RAM.
pace = load_pace_l2_aop(
    PACE_PATH,
    expected_sha256=sources['src.pace-oci-20240730-nrt']['sha256'],
)
active_exclude = tuple(f for f in DEFAULT_PACE_EXCLUDE_FLAGS if f in pace.flag_definitions)
skipped_flags = [f for f in DEFAULT_PACE_EXCLUDE_FLAGS if f not in pace.flag_definitions]
assert active_exclude, 'No recognized PACE l2_flags found in granule attributes; refusing unscreened swath.'
if skipped_flags:
    print('Flag names absent from granule attributes (skipped):', skipped_flags)

flag_res = decode_l2_flags(pace.l2_flags, pace.flag_definitions, exclude_flags=active_exclude)
valid_mask = flag_res['valid_mask']

rgb_vis, vis_wl = build_rgb_composite(
    pace.rrs, pace.wavelengths_nm, rgb_wavelengths_nm=(665.0, 550.0, 442.0), valid_mask=valid_mask
)
rgb_false, false_wl = build_rgb_composite(
    pace.rrs, pace.wavelengths_nm, rgb_wavelengths_nm=(704.0, 620.0, 550.0), valid_mask=valid_mask
)
pca = compute_swath_pca(
    pace.rrs, pace.wavelengths_nm, valid_mask=valid_mask, min_nm=400.0, max_nm=719.0, n_components=3
)

# Compute PACE trapezoidal AVW (PACE OCI wavelength_3d is non-uniform across ~600 nm) and CIE colour.
pace_ok = pca['valid_mask_used']
pace_valid_rrs = pace.rrs[pace_ok]
pace_avw_nm = compute_avw(
    pace_valid_rrs, pace.wavelengths_nm, integration='trapezoid', require_nonnegative=True
)
pace_xyz = rrs_to_xyz(pace_valid_rrs, pace.wavelengths_nm, illuminant='E')
pace_xy = xyz_to_chromaticity(pace_xyz)
pace_hue_deg = chromaticity_to_hue_angle_deg(pace_xy)
pace_srgb = xyz_to_display_srgb(pace_xyz, illuminant='E', exposure='max_y', gamut='desaturate')

print('Visible RGB wavelengths (nm):', vis_wl)
print('False-colour wavelengths (nm):', false_wl)
print('Swath PCA explained variance ratio (PC1-PC3):', np.round(pca['explained_variance_ratio'], 4))
print('PACE valid pixels:', int(pace_ok.sum()), '| finite AVW:', int(np.isfinite(pace_avw_nm).sum()), '| finite CIE hue:', int(np.isfinite(pace_hue_deg).sum()))
print('Swath pixel accounting:', pca['counts'].to_dict())

## 3. Exploratory GLORIA AVW-quartile optical archetypes & site-aggregated SAM OOD bridge

We align non-negative GLORIA spectra onto PACE OCI 400–700 nm band centers via linear interpolation (exploratory alignment, not official OCI SRF convolution), build four quartile optical archetypes for `classify_owt_by_sam` (distinct from published Spyrakos 13-type or `pyOWT` 10-type libraries), and aggregate GLORIA spectra by `Country|Site_name` median before `nearest_neighbor_sam_ood` so same-site replicates do not compress the internal LOO SAM threshold.

In [ ]:
pace_vis_mask = (pace.wavelengths_nm >= 400.0) & (pace.wavelengths_nm <= 700.0)
wl_common = pace.wavelengths_nm[pace_vis_mask]

# Filter to non-negative GLORIA spectra with valid AVW and site keys.
site_keys_all, site_valid_all = build_site_group_keys(gloria.meta)
site_keys_vis = site_keys_all[keep_vis]
site_valid_vis = site_valid_all[keep_vis]
gloria_ref_mask = np.isfinite(avw_nm) & (np.linalg.norm(X_vis, axis=1) > 0.0) & site_valid_vis
assert int(gloria_ref_mask.sum()) >= 4, 'Too few valid GLORIA reference spectra for optical archetypes.'

X_gloria_common = np.vstack([
    np.interp(wl_common, wl_gloria, row) for row in X_vis[gloria_ref_mask]
])
avw_ref = avw_nm[gloria_ref_mask]
sites_ref = site_keys_vis[gloria_ref_mask]

# Four exploratory AVW-quartile median archetypes on the PACE 400-700 nm grid.
q_edges = np.quantile(avw_ref, [0.0, 0.25, 0.50, 0.75, 1.0])
archetype_labels = ['Q1_bluest_AVW', 'Q2_blue_green_AVW', 'Q3_green_AVW', 'Q4_reddest_AVW']
archetype_spectra = []
for i in range(4):
    lo_q, hi_q = q_edges[i], q_edges[i + 1]
    in_q = (avw_ref >= lo_q) & (avw_ref <= hi_q if i == 3 else avw_ref < hi_q)
    archetype_spectra.append(np.median(X_gloria_common[in_q], axis=0))
archetype_matrix = np.vstack(archetype_spectra)

# Site-level median GLORIA spectra for replicate-safe LOO SAM OOD calibration.
site_df = pd.DataFrame(X_gloria_common).assign(site=sites_ref).groupby('site', sort=True).median()
X_gloria_site_medians = site_df.to_numpy(dtype=float)
assert X_gloria_site_medians.shape[0] >= 2, 'Need >= 2 distinct GLORIA sites for SAM OOD threshold.'

# Sample up to 2,000 valid PACE pixels for interactive screening.
pace_specs = pace_valid_rrs[:, pace_vis_mask]
rng = np.random.default_rng(2026)
n_sample = min(2000, pace_specs.shape[0])
sample_idx = rng.choice(pace_specs.shape[0], size=n_sample, replace=False)
pace_sample = pace_specs[sample_idx]

ood = nearest_neighbor_sam_ood(X_gloria_site_medians, pace_sample, ood_quantile=0.95)
owt = classify_owt_by_sam(
    pace_sample,
    archetype_matrix,
    archetype_labels,
    wavelengths_nm=wl_common,
    max_angle_rad=ood['threshold_sam_rad'],
)
print(
    f'Site-aggregated GLORIA ({len(X_gloria_site_medians)} sites) 95th-pct LOO SAM threshold: '
    f'{ood["threshold_sam_rad"]:.4f} rad | PACE sampled OOD: {int(ood["is_ood"].sum())}/{n_sample}'
)
print('Exploratory archetype counts:', pd.Series(owt['assigned_label']).value_counts(dropna=False).to_dict())

# Record current git commit SHA when executing inside a Git checkout.
git_proc = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, capture_output=True, text=True, encoding='utf-8', check=False)
git_commit = git_proc.stdout.strip() if git_proc.returncode == 0 and len(git_proc.stdout.strip()) == 40 else None

gloria_counts = CohortCounts(
    attempted=n_total_gloria,
    completed=int(keep_vis.sum()),
    failed=0,
    excluded=int(n_total_gloria - keep_vis.sum()),
    exclusion_reasons={
        'incomplete_400_700nm': n_incomplete_vis,
        'qc_flagged': n_qc_flagged_complete,
    },
)
manifest = build_manifest(
    experiment_id='Q01',
    title='GLORIA CIE water colour atlas and PACE-01 optical exploration',
    status='exploratory',
    git_commit=git_commit,
    script_path='notebooks/Q01_water_colour_and_owts.ipynb',
    script_sha256=sha256_file(REPO_ROOT / 'notebooks' / 'Q01_water_colour_and_owts.ipynb'),
    inputs={
        'src.gloria-2022': gloria.archive_sha256,
        'src.pace-oci-20240730-nrt': pace.file_sha256,
    },
    counts={'gloria_visible': gloria_counts, 'pace_swath_pca': pca['counts']},
    units={'Rrs': 'sr^-1', 'avw_nm': 'nm', 'hue_angle_deg': 'deg', 'sam_rad': 'rad'},
    random_seed=2026,
    pairing_rule='Exploratory linear interpolation of site-median GLORIA spectra onto PACE OCI 400-700 nm band centers (not official OCI SRF convolution)',
    extra={
        'n_gloria_site_medians': int(len(X_gloria_site_medians)),
        'n_pace_ood_sampled': int(n_sample),
        'n_pace_ood_flagged': int(ood['is_ood'].sum()),
        'sam_ood_threshold_rad': float(ood['threshold_sam_rad']),
    },
)
write_manifest(manifest, OUT_DIR / 'manifest.json')
print('Wrote deterministic manifest to:', OUT_DIR / 'manifest.json')